# Freeman-Adelson G1 step-edge example

A first-order steerable Gaussian derivative basis. This notebook creates a simple black-to-white step edge, runs the filter, and plots the result. The code is intentionally short so you can change the parameters and rerun the cells.


Import the small set of tools used in this notebook. The autoreload lines help Jupyter pick up local source edits without restarting the kernel every time.


In [ ]:
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import torch

from agfb_filters import ExecutionPath, get_filter_definition, run_filter

Choose the filter settings here. The `definition` describes the filter, and the `path` chooses the matching execution path.


In [ ]:
sigma = 1.0
definition = get_filter_definition("freeman_adelson_g1", sigma=sigma)
path = ExecutionPath.SEPARABLE

Create a test image. By default this is a synthetic step edge. The commented block lets you replace it with your own image file path.


In [ ]:
size = 1024
image = torch.zeros(1, size, size)
image[:, :, size // 2 :] = 1.0

# To use your own image instead, uncomment this block and set image_path.
# image_path = "path/to/image.png"
# image = torch.as_tensor(plt.imread(image_path), dtype=torch.float32)
# if image.ndim == 3:
#     image = image[..., :3].mean(dim=-1)
# image = image.unsqueeze(0)

plt.figure(figsize=(4, 4))
plt.imshow(image[0], cmap="gray", vmin=0, vmax=1)
plt.title("input image")
plt.axis("off")

Visualize the fixed filter weights. `kernel_x` responds to left-to-right changes, and `kernel_y` responds to top-to-bottom changes.


In [ ]:
kernel_x, kernel_y = definition.dense_kernels()
weight_limit = max(float(kernel_x.abs().max()), float(kernel_y.abs().max()))

fig, axes = plt.subplots(1, 2, figsize=(8, 3))

for ax, kernel, title in zip(
    axes,
    (kernel_x, kernel_y),
    ("kernel_x", "kernel_y"),
    strict=False,
):
    ax.imshow(kernel, cmap="gray", vmin=-weight_limit, vmax=weight_limit)
    ax.set_title(title)
    ax.axis("off")

fig.suptitle(f"{definition.name} weights")
fig.tight_layout()

Run the filter. The timer measures only the filter call, not the image setup or plotting.


In [ ]:
start = time.perf_counter()
gradient_x, gradient_y = run_filter(
    definition,
    image,
    path=path,
    boundary=definition.default_boundary,
)
elapsed = time.perf_counter() - start

print(f"{elapsed:.4f} seconds")

Plot the two gradient outputs with the same grayscale range. The input changes left-to-right, so the strongest response should usually be in `gradient_x`.


In [ ]:
limit = max(float(gradient_x.abs().max()), float(gradient_y.abs().max()))

fig, axes = plt.subplots(1, 2, figsize=(8, 3))

axes[0].imshow(gradient_x[0], cmap="gray", vmin=-limit, vmax=limit)
axes[0].set_title("gradient_x")

axes[1].imshow(gradient_y[0], cmap="gray", vmin=-limit, vmax=limit)
axes[1].set_title("gradient_y")

for ax in axes:
    ax.axis("off")

fig.suptitle(f"{definition.name}, time={elapsed:.4f} s")
fig.tight_layout()